In [75]:
import pandas as pd 
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import grangercausalitytests
import numpy as np
import statsmodels.api as sm
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from scipy.stats import pearsonr

**Data cleaning**

In [76]:
data = pd.read_csv("C:\\Users\\王亭烜\\Desktop\\Thesis\\Data\\data1221\\EstimatedStates1.csv")
sp_data = pd.read_csv("C:\\Users\\王亭烜\\Desktop\\Thesis\\Data\\data1221\\sp_raw_data.csv")
rf_data = pd.read_excel("C:\\Users\\王亭烜\\Desktop\\Thesis\\Data\\data1221\\rf_raw_data.xlsx")

rf_data['observation_date'] =pd.to_datetime(rf_data['observation_date'])
sp_data['caldt'] = pd.to_datetime(sp_data['caldt'])
rf_filtered_data = rf_data[(rf_data['observation_date'] >= '2010-06-01') & (rf_data['observation_date'] <= '2023-08-31')]
rf_filtered_data = rf_filtered_data.set_index('observation_date').resample('M').last().reset_index()
rf_filtered_data['DTB3'] = rf_filtered_data['DTB3'] / (100*12)
filtered_data = sp_data[(sp_data['caldt'] >= '2010-06-01') & (sp_data['caldt'] <= '2023-08-31')]
filtered_data = filtered_data.set_index('caldt').resample('M').last().reset_index()
final_df = pd.concat([data.reset_index(drop=True), filtered_data.reset_index(drop=True)], axis=1)
final_df = pd.concat([final_df, rf_filtered_data], axis=1)
final_df['log_return'] = np.log(final_df['spindx'] / final_df['spindx'].shift(1))
final_df['ExcessReturn'] = final_df['log_return'] - final_df['DTB3']
final_df['log_return_shifted'] = final_df['ExcessReturn'].shift(-1)
final_df 

C:\Users\王亭烜\AppData\Local\Temp\ipykernel_15784\259553903.py:8: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  rf_filtered_data = rf_filtered_data.set_index('observation_date').resample('M').last().reset_index()
C:\Users\王亭烜\AppData\Local\Temp\ipykernel_15784\259553903.py:11: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  filtered_data = filtered_data.set_index('caldt').resample('M').last().reset_index()


,Date,YearMonth,Alpha,caldt,vwretd,vwretx,ewretd,ewretx,totval,totcnt,usdval,usdcnt,spindx,sprtrn,observation_date,DTB3,log_return,ExcessReturn,log_return_shifted
0,2010-06-30,2010-06-01,1.109127,2010-06-30,-0.010014,-0.010128,-0.010721,-0.010795,9.545851e+09,500,9.647394e+09,500,1030.71,-0.010113,2010-06-30,0.000150,NaN,NaN,0.066391
1,2010-07-28,2010-07-01,1.144103,2010-07-31,0.000052,0.000052,0.001221,0.001221,1.019212e+10,500,1.020835e+10,500,1101.60,0.000064,2010-07-31,0.000125,0.066516,0.066391,-0.048728
2,2010-08-25,2010-08-01,0.974654,2010-08-31,-0.000067,-0.000110,-0.001044,-0.001068,9.714332e+09,500,9.722384e+09,500,1049.33,0.000391,2010-08-31,0.000117,-0.048612,-0.048728,0.083795
3,2010-09-29,2010-09-01,0.969009,2010-09-30,-0.003101,-0.003106,-0.001975,-0.001986,1.056521e+10,500,1.060877e+10,500,1141.20,-0.003084,2010-09-30,0.000133,0.083928,0.083795,0.036093
4,2010-10-27,2010-10-01,0.890464,2010-10-31,-0.000142,-0.000142,0.003006,0.003006,1.095012e+10,500,1.095918e+10,500,1183.26,-0.000439,2010-10-31,0.000100,0.036193,0.036093,-0.002435
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
154,2023-04-26,2023-04-01,0.932890,2023-04-30,0.008069,0.007955,0.010616,0.010409,3.625873e+10,503,3.598892e+10,503,4169.48,0.008253,2023-04-30,0.004125,0.014536,0.010411,-0.001904
155,2023-05-31,2023-05-01,0.917655,2023-05-31,-0.005681,-0.005884,-0.009001,-0.009284,3.637285e+10,503,3.661214e+10,503,4179.83,-0.006109,2023-05-31,0.004383,0.002479,-0.001904,0.058411
156,2023-06-28,2023-06-01,0.756118,2023-06-30,0.012340,0.012304,0.009344,0.009258,3.879290e+10,503,3.835421e+10,503,4450.38,0.012269,2023-06-30,0.004308,0.062719,0.058411,0.026264
157,2023-07-26,2023-07-01,0.726123,2023-07-31,0.001518,0.001490,0.002962,0.002844,3.996646e+10,503,3.991144e+10,503,4588.96,0.001469,2023-07-31,0.004400,0.030664,0.026264,-0.022309


In [77]:
gw_data = pd.read_csv("C:\\Users\\王亭烜\\Downloads\\return prediction(1).csv")
gw_data['date'] = pd.to_datetime(gw_data['yyyymm'].astype(str), format='%Y%m')
start_date = "2010-06-01" 
end_date = "2023-08-31" 
gw_filtered_data = gw_data[(gw_data['date'] >= start_date) & (gw_data['date'] <= end_date)]
gw_filtered_data = gw_filtered_data.reset_index(drop=True)

final_df = pd.concat([final_df, gw_filtered_data], axis=1)
final_df.columns 

Index(['Date', 'YearMonth', 'Alpha', 'caldt', 'vwretd', 'vwretx', 'ewretd',
       'ewretx', 'totval', 'totcnt', 'usdval', 'usdcnt', 'spindx', 'sprtrn',
       'observation_date', 'DTB3', 'log_return', 'ExcessReturn',
       'log_return_shifted', 'Unnamed: 0', 'yyyymm', 'b/m', 'tbl', 'lty',
       'ntis', 'infl', 'ltr', 'svar', 'ep', 'epx', 'log_dp', 'log_dy',
       'log_ep', 'log_de', 'tms', 'dfy', 'dfs', 'date'],
      dtype='object')

**Univariate & bivariate IS regression**

In [96]:
regression_data = final_df.dropna(subset=['log_return_shifted', 'Alpha', 'dfs'])
scaler = StandardScaler()
regression_data.loc[:, 'Alpha_scaled'] = scaler.fit_transform(regression_data[['Alpha']])[:, 0]
regression_data.loc[:, 'Predictor_scaled'] = scaler.fit_transform(regression_data[['dfs']])[:, 0]
X = regression_data[['Alpha_scaled', 'Predictor_scaled']]
y = regression_data['log_return_shifted']
X = sm.add_constant(X) 

model = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 1}) 
print("Full Model with Newey-West Standard Errors:")
print(model.summary())


Full Model with Newey-West Standard Errors:
                            OLS Regression Results                            
Dep. Variable:     log_return_shifted   R-squared:                       0.024
Model:                            OLS   Adj. R-squared:                  0.012
Method:                 Least Squares   F-statistic:                     2.258
Date:                Fri, 07 Mar 2025   Prob (F-statistic):              0.108
Time:                        01:55:14   Log-Likelihood:                 279.20
No. Observations:                 158   AIC:                            -552.4
Df Residuals:                     155   BIC:                            -543.2
Df Model:                           2                                         
Covariance Type:                  HAC                                         
                       coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------

C:\Users\王亭烜\AppData\Local\Temp\ipykernel_15784\2424125971.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  regression_data.loc[:, 'Alpha_scaled'] = scaler.fit_transform(regression_data[['Alpha']])[:, 0]
C:\Users\王亭烜\AppData\Local\Temp\ipykernel_15784\2424125971.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  regression_data.loc[:, 'Predictor_scaled'] = scaler.fit_transform(regression_data[['dfy']])[:, 0]


**Summary statistics**

In [86]:
predictors = ['Alpha', 'b/m', 'tbl', 'lty', 'ntis', 'infl', 'ltr', 'svar', 'log_dp', 'log_dy', 'log_ep', 'log_de', 'tms', 'dfy', 'dfs']

for predictor in predictors:
    mean = final_df[predictor].mean()
    std = final_df[predictor].std() 
    min_val = final_df[predictor].min() 
    median = final_df[predictor].median() 
    max_val = final_df[predictor].max() 
    print(f"{predictor:<10} | Mean: {mean:>8.4f} | Std: {std:>8.4f} | Min: {min_val:>8.4f} | Median: {median:>8.4f} | Max: {max_val:>8.4f}")
    

Alpha      | Mean:   1.0384 | Std:   0.2813 | Min:   0.6368 | Median:   0.9795 | Max:   2.6716
b/m        | Mean:   0.2854 | Std:   0.0602 | Min:   0.1756 | Median:   0.2935 | Max:   0.4242
tbl        | Mean:   0.0086 | Std:   0.0131 | Min:   0.0001 | Median:   0.0014 | Max:   0.0530
lty        | Mean:   0.0262 | Std:   0.0086 | Min:   0.0062 | Median:   0.0265 | Max:   0.0432
ntis       | Mean:  -0.0065 | Std:   0.0141 | Min:  -0.0326 | Median:  -0.0097 | Max:   0.0201
infl       | Mean:   0.0021 | Std:   0.0035 | Min:  -0.0067 | Median:   0.0019 | Max:   0.0137
ltr        | Mean:   0.0037 | Std:   0.0290 | Min:  -0.0629 | Median:  -0.0003 | Max:   0.0862
svar       | Mean:   0.0026 | Std:   0.0061 | Min:   0.0002 | Median:   0.0013 | Max:   0.0732
log_dp     | Mean:  -3.9836 | Std:   0.1314 | Min:  -4.3684 | Median:  -3.9496 | Max:  -3.7700
log_dy     | Mean:  -3.9747 | Std:   0.1317 | Min:  -4.3597 | Median:  -3.9387 | Max:  -3.7694
log_ep     | Mean:  -3.0517 | Std:   0.2135 | Min: